In [148]:
#Importar bibliotecas
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [149]:
#Ler o CSV limpo 
df = pd.read_csv('../data/nhl_draft_clean.csv')
df.head()

,year,overall_pick,round,team,player,player_ep_id,position,nationality,amateur_league,seasons_pre_nhl,games_played,goals,assists,points,penalty_minutes,points_per_game,goals_per_game
0,2015,1,1,Edmonton Oilers,Connor McDavid,183442,F,CAN,OHL,3.0,166.0,97.0,188.0,285.0,104.0,1.72,0.58
1,2015,2,1,Buffalo Sabres,Jack Eichel,191959,F,USA,USDP,5.0,210.0,126.0,156.0,282.0,120.0,1.34,0.60
2,2015,3,1,Arizona Coyotes,Dylan Strome,228107,F,CAN,OHL,4.0,219.0,114.0,240.0,354.0,105.0,1.62,0.52
3,2015,4,1,Toronto Maple Leafs,Mitchell Marner,223194,F,CAN,OHL,3.0,184.0,96.0,205.0,301.0,145.0,1.64,0.52
4,2015,5,1,Carolina Hurricanes,Noah Hanifin,177710,D,USA,USHS-Prep,6.0,210.0,38.0,124.0,162.0,72.0,0.77,0.18


In [150]:
#Criar o target
df['target'] = (df['games_played'] > 100).astype(int)
df[['games_played', 'target']].head()

,games_played,target
0,166.0,1
1,210.0,1
2,219.0,1
3,184.0,1
4,210.0,1


In [151]:
#Definir as features
features = ['amateur_league', 'seasons_pre_nhl', 'points_per_game', 'penalty_minutes']
X = df[features] 
y = df['target'] 

In [152]:
#Indentificar colunas categóricas e numéricas
categorical_features = ['amateur_league']
numeric_features = ['seasons_pre_nhl', 'points_per_game', 'penalty_minutes']

In [153]:
#Preparar a transformação de dados
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numeric_features)
    ]
)

In [154]:
# Dividir em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2, # 20% para teste
    random_state=42
)

In [155]:
#Criar a pipeline com o modelo
model = Pipeline(
    steps =[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(random_state=42))
    ]
)

In [156]:
#Treinar o modelo
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [157]:
#Fazer previsões
y_proba = model.predict_proba(X_test)[:, 1]
threshold = 0.3
y_pred = (y_proba >= threshold).astype(int)

In [158]:
#Avaliar
print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9652777777777778

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.61      0.75        36
           1       0.97      1.00      0.98       396

    accuracy                           0.97       432
   macro avg       0.96      0.80      0.86       432
weighted avg       0.96      0.97      0.96       432


Confusion Matrix:
[[ 22  14]
 [  1 395]]


In [159]:
#Criar tabela de comparação
results_df = df.loc[X_test.index, ['player', 'games_played']].copy()
results_df['real'] = y_test
results_df['predicted'] = y_pred
results_df.head()

,player,games_played,real,predicted
746,Luke Loheit,214.0,1,1
1677,Albert Wikman,215.0,1,1
1858,Chase Pietila,240.0,1,1
1657,Ty Mueller,98.0,0,0
479,Tim Söderlund,467.0,1,1


In [160]:
#Ver onde o modelo errou (errou prevendo 1 quando era 0)
false_positives = results_df[(results_df['real'] == 0) & (results_df['predicted'] == 1)]
false_positives

,player,games_played,real,predicted
1356,Simon Nemec,88.0,0,1
1302,Ben Boyd,75.0,0,1
637,Jan Jeník,70.0,0,1
571,Kristian Røykås Marthinsen,65.0,0,1
1247,Gannon Laroque,91.0,0,1
1374,Ivan Miroshnichenko,56.0,0,1
782,Ville Heinola,82.0,0,1
1050,Carter Savoie,89.0,0,1
2122,Jan Chovan,60.0,0,1
1912,Charlie Forslund,90.0,0,1


In [161]:
#Ver onde o modelo errou (errou prevendo 0 quando era 1)
false_negatives = results_df[(results_df['real'] == 1) & (results_df['predicted'] == 0)]
false_negatives

,player,games_played,real,predicted
1419,Miko Matikka,101.0,1,0


In [162]:
#Ver acertos (classe 0)
true_negatives = results_df[(results_df['real'] == 0) & (results_df['predicted'] == 0)]
true_negatives

,player,games_played,real,predicted
1657,Ty Mueller,98.0,0,0
2045,Maxim Schäfer,84.0,0,0
1731,Janne Peltonen,20.0,0,0
974,Lukas Reichel,80.0,0,0
2113,Victor Hedin Raftheim,87.0,0,0
765,Kaapo Kakko,51.0,0,0
383,Cale Makar,75.0,0,0
1779,Stian Solberg,47.0,0,0
188,Patrik Laine,52.0,0,0
1770,Konsta Helenius,84.0,0,0


In [163]:
#Ver acertos (classe 1)
true_positives = results_df[(results_df['real'] == 1) & (results_df['predicted'] == 1)]
true_positives

,player,games_played,real,predicted
746,Luke Loheit,214.0,1,1
1677,Albert Wikman,215.0,1,1
1858,Chase Pietila,240.0,1,1
479,Tim Söderlund,467.0,1,1
1730,Oiva Keskinen,162.0,1,1
...,...,...,...,...
1237,Taige Harding,123.0,1,1
1590,Mikhail Gulyayev,274.0,1,1
2040,Brady Peddle,124.0,1,1
1916,Trent Swick,203.0,1,1


In [164]:
true_negatives.head(10)

,player,games_played,real,predicted
1657,Ty Mueller,98.0,0,0
2045,Maxim Schäfer,84.0,0,0
1731,Janne Peltonen,20.0,0,0
974,Lukas Reichel,80.0,0,0
2113,Victor Hedin Raftheim,87.0,0,0
765,Kaapo Kakko,51.0,0,0
383,Cale Makar,75.0,0,0
1779,Stian Solberg,47.0,0,0
188,Patrik Laine,52.0,0,0
1770,Konsta Helenius,84.0,0,0


In [165]:
true_positives.head(10)

,player,games_played,real,predicted
746,Luke Loheit,214.0,1,1
1677,Albert Wikman,215.0,1,1
1858,Chase Pietila,240.0,1,1
479,Tim Söderlund,467.0,1,1
1730,Oiva Keskinen,162.0,1,1
1452,Tyson Jugnauth,151.0,1,1
905,Braden Doyle,271.0,1,1
605,Joe Veleno,276.0,1,1
670,Nico Gross,427.0,1,1
1508,Theo Wallberg,211.0,1,1


In [166]:
false_positives.head(10)

,player,games_played,real,predicted
1356,Simon Nemec,88.0,0,1
1302,Ben Boyd,75.0,0,1
637,Jan Jeník,70.0,0,1
571,Kristian Røykås Marthinsen,65.0,0,1
1247,Gannon Laroque,91.0,0,1
1374,Ivan Miroshnichenko,56.0,0,1
782,Ville Heinola,82.0,0,1
1050,Carter Savoie,89.0,0,1
2122,Jan Chovan,60.0,0,1
1912,Charlie Forslund,90.0,0,1


In [167]:
false_negatives.head(10)

,player,games_played,real,predicted
1419,Miko Matikka,101.0,1,0


In [168]:
len(true_negatives)

22

In [169]:
len(true_positives)

395

In [170]:
len(false_negatives)

1

In [171]:
len(false_positives)

14

In [172]:
from sklearn.metrics import precision_score, recall_score, f1_score

results = []
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_t = (y_proba >= t).astype(int)
    results.append({
        'threshold': t,
        'accuracy': accuracy_score(y_test, y_pred_t),
        'precision_0': precision_score(y_test, y_pred_t, pos_label=0),
        'recall_0': recall_score(y_test, y_pred_t, pos_label=0),
        'precision_1': precision_score(y_test, y_pred_t, pos_label=1),
        'recall_1': recall_score(y_test, y_pred_t, pos_label=1),
        'f1_1': f1_score(y_test, y_pred_t, pos_label=1),
    })

threshold_df = pd.DataFrame(results)
threshold_df

,threshold,accuracy,precision_0,recall_0,precision_1,recall_1,f1_1
0,0.1,0.937500,1.000000,0.250000,0.936170,1.000000,0.967033
1,0.2,0.946759,0.933333,0.388889,0.947242,0.997475,0.971710
2,0.3,0.965278,0.956522,0.611111,0.965770,0.997475,0.981366
3,0.4,0.967593,0.892857,0.694444,0.972772,0.992424,0.982500
4,0.5,0.962963,0.777778,0.777778,0.979798,0.979798,0.979798
5,0.6,0.958333,0.714286,0.833333,0.984615,0.969697,0.977099
6,0.7,0.956019,0.697674,0.833333,0.984576,0.967172,0.975796
